In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# 1. Users (sample combined user list)
users_data = [
    {"user_id": 1, "name": "User_1", "city": "Chennai", "membership": "Regular"},
    {"user_id": 2, "name": "User_2", "city": "Pune", "membership": "Gold"},
    {"user_id": 3, "name": "User_3", "city": "Hyderabad", "membership": "Gold"},
    {"user_id": 4, "name": "User_4", "city": "Bengaluru", "membership": "Regular"},
]
df_users = pd.DataFrame(users_data)

# 2. Restaurants (sample)
restaurants_data = [
    [1, "Restaurant_1", "Chinese", 4.8],
    [2, "Restaurant_2", "Indian", 4.1],
    [3, "Restaurant_3", "Italian", 4.6],
    [4, "Restaurant_4", "Mexican", 4.4],
]
df_restaurants = pd.DataFrame(restaurants_data, columns=['res_id', 'name', 'cuisine', 'rating'])

# 3. Create a sample orders.csv if it doesn't exist
orders_path = 'orders.csv'
if not os.path.exists(orders_path):
    orders = [
        {'order_id': 101, 'user_id': 1, 'res_id': 1, 'amount': 250, 'city': 'Chennai'},
        {'order_id': 102, 'user_id': 2, 'res_id': 2, 'amount': 400, 'city': 'Pune'},
        {'order_id': 103, 'user_id': 3, 'res_id': 3, 'amount': 320, 'city': 'Hyderabad'},
        {'order_id': 104, 'user_id': 2, 'res_id': 3, 'amount': 150, 'city': 'Pune'},
        {'order_id': 105, 'user_id': 4, 'res_id': 4, 'amount': 180, 'city': 'Bengaluru'},
        {'order_id': 106, 'user_id': 3, 'res_id': 1, 'amount': 270, 'city': 'Hyderabad'},
        {'order_id': 107, 'user_id': 1, 'res_id': 2, 'amount': 220, 'city': 'Chennai'},
        {'order_id': 108, 'user_id': 3, 'res_id': 3, 'amount': 500, 'city': 'Hyderabad'},
    ]
    df_temp = pd.DataFrame(orders)
    df_temp.to_csv(orders_path, index=False)
    print('Created sample orders.csv')

# 4. Load orders
df_orders = pd.read_csv(orders_path)

# Merge data for comprehensive analysis
df_combined = df_orders.merge(df_users, on='user_id', how='left').merge(df_restaurants, left_on='res_id', right_on='res_id', how='left')

# Calculations
gold_orders_count = df_combined[df_combined['membership'] == 'Gold'].shape[0]
hyd_revenue = round(df_orders[df_orders['city'] == 'Hyderabad']['amount'].sum())
distinct_users = df_orders['user_id'].nunique()
aov_gold = round(df_combined[df_combined['membership'] == 'Gold']['amount'].mean(), 2)
high_rating_orders = df_combined[df_combined['rating'] >= 4.5].shape[0]

# Top revenue city for Gold members (the merge may create city_x for orders and city_y for users)
order_city_col = 'city_x' if 'city_x' in df_combined.columns else 'city'
top_city_gold = df_combined[df_combined['membership'] == 'Gold'].groupby(order_city_col)['amount'].sum().idxmax()
top_city_orders = df_combined[(df_combined['membership'] == 'Gold') & (df_combined[order_city_col] == top_city_gold)].shape[0]

print(f"Gold Orders: {gold_orders_count}")
print(f"Hyd Revenue: {hyd_revenue}")
print(f"Distinct Users: {distinct_users}")
print(f"Gold AOV: {aov_gold}")
print(f"Rating >= 4.5 Orders: {high_rating_orders}")
print(f"Top City (Gold) Orders: {top_city_orders}")

# --- PLOTTING ---
# 1) Revenue by City
plt.figure(figsize=(8,5))
rev_by_city = df_orders.groupby('city')['amount'].sum().sort_values(ascending=False)
sns.barplot(x=rev_by_city.index, y=rev_by_city.values, palette='Blues_d')
plt.title('Revenue by City')
plt.ylabel('Revenue')
plt.xlabel('City')
plt.tight_layout()
plt.savefig('revenue_by_city.png')
plt.show()

# 2) Revenue by Restaurant (top)
plt.figure(figsize=(8,5))
rev_by_res = df_combined.groupby('name')['amount'].sum().sort_values(ascending=False)
sns.barplot(x=rev_by_res.values, y=rev_by_res.index, palette='viridis')
plt.title('Revenue by Restaurant')
plt.xlabel('Revenue')
plt.ylabel('Restaurant')
plt.tight_layout()
plt.savefig('revenue_by_restaurant.png')
plt.show()

# 3) Average Order Value by Membership
plt.figure(figsize=(6,4))
aov = df_combined.groupby('membership')['amount'].mean().round(2)
sns.barplot(x=aov.index, y=aov.values, palette='pastel')
plt.title('Average Order Value by Membership')
plt.ylabel('AOV')
plt.xlabel('Membership')
plt.tight_layout()
plt.savefig('aov_by_membership.png')
plt.show()

# 4) Histogram of Order Amounts
plt.figure(figsize=(6,4))
sns.histplot(df_orders['amount'], bins=8, kde=False, color='coral')
plt.title('Order Amount Distribution')
plt.xlabel('Amount')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('order_amount_hist.png')
plt.show()

print('Saved plots: revenue_by_city.png, revenue_by_restaurant.png, aov_by_membership.png, order_amount_hist.png')